# Sudoku-1 : Résolution par Backtracking

**Navigation** : [<< Sudoku-0 Environment](Sudoku-0-Environment-Csharp.ipynb) | [Index](README.md) | [Sudoku-3 Genetic >>](Sudoku-3-Genetic-Csharp.ipynb)

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :
1. **Implémenter** un algorithme de backtracking pour résoudre le Sudoku
2. **Comprendre** l'exploration en profondeur et le retour arrière
3. **Analyser** les performances du backtracking selon la difficulté des puzzles

**Durée estimée** : ~7 min | **Prérequis** : [Sudoku-0 Environment](Sudoku-0-Environment-Csharp.ipynb) | **Lien** : Voir [CSP-1-Fondamentaux](../Search/Part2-CSP/CSP-1-Fundamentals.ipynb) pour la théorie du backtracking

---

## Introduction Théorique

L'algorithme de backtracking est une méthode de recherche en profondeur utilisée pour résoudre les problèmes de satisfaction de contraintes (CSP), comme le Sudoku. L'algorithme explore toutes les configurations possibles pour trouver une solution qui respecte les contraintes :

- **Exploration en profondeur** : L'algorithme explore chaque possibilité de manière exhaustive avant de revenir en arrière (backtrack) lorsque aucune solution n'est trouvée dans une branche particulière.
- **Contraintes** : Dans le cas du Sudoku, les contraintes sont les règles du jeu : chaque chiffre de 1 à 9 doit apparaître une seule fois par ligne, colonne et sous-grille de 3x3.

## Implémentation de l'Algorithme de Backtracking

L'algorithme suit ces étapes :
1. Trouver une case vide dans la grille.
2. Tenter de placer un chiffre (1-9) dans la case vide.
3. Vérifier si ce chiffre respecte les contraintes.
4. Si oui, passer a la case suivante et répéter le processus.
5. Si non, essayer le chiffre suivant.
6. Si aucun chiffre ne convient, revenir en arrière (backtrack).

## Importation des Classes de Base


In [1]:
#!import Sudoku-0-Environment-Csharp.ipynb


# Sudoku-0 : Environnement et Classes de Base (C#)

**Navigation** : [Index](README.md) | [Sudoku-1 Backtracking C# >>](Sudoku-1-Backtracking-Csharp.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Comprendre la structure de donnees `SudokuGrid` et ses methodes principales
2. Utiliser `ISudokuSolver` pour implementer un solveur de Sudoku
3. Exploiter `SudokuHelper` pour charger des grilles et tester des solveurs
4. Comparer les performances de plusieurs solveurs sur differentes difficultes

**Prerequis** : Notions de base en C# (.NET Interactive)  
**Duree estimee** : ~15 min

Installed Packages Plotly.NET, 5.1.0

## Définition de la classe SudokuGrid

Nous définissons ici la classe SudokuGrid qui représente une grille de Sudoku et fournit des méthodes pour manipuler et afficher les grilles.


SudokuGrid defini.


### Interpretation : Structure de donnees pour la grille Sudoku

**Sortie obtenue** : La classe `SudokuGrid` encapsule toutes les operations de manipulation, validation et affichage d'une grille de Sudoku 9x9.

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| `Cells[9,9]` | int[,] | Stockage interne des valeurs (0 = vide) |
| `AllNeighbours` | 27 x 9 positions | Pre-calcul des voisins ligne/colonne/bloc |
| `CellNeighbours[9][9]` | ~20 positions chacune | Voisins directs de chaque cellule |
| `GetAvailableNumbers()` | int[] | Candidats valides pour une cellule |
| `NbErrors()` | int | Nombre de conflits + modifications erronees |

**Points cles** :
1. **Pre-calcul des voisins** : `AllNeighbours` et `CellNeighbours` sont calcules une seule fois a l'initialisation, evitant les recalculs couteux
2. **Conversion flexible** : Methodes pour convertir entre tableaux 1D, 2D et jagged arrays (utile pour differents formats de fichiers)
3. **Validation robuste** : `NbErrors` compte a la fois les doublons (ligne/colonne/bloc) et les modifications de indices pre-remplis
4. **Parsing tolerant** : `ReadMultiSudoku` accepte plusieurs formats (`.`, `X`, `-`, espaces)

> **Note technique** : La structure `CellNeighbours[i][j]` contient environ 20 positions (8 ligne + 8 colonne + 4 bloc, moins les doublons). Ce pre-calcul est crucial pour les performances des algorithmes de backtracking et de propagation de contraintes.

...

SudokuHelper defini.


### Interpretation : Infrastructure de test et benchmark

**Sortie obtenue** : La classe `SudokuHelper` fournit une infrastructure complete pour tester et comparer les solveurs de Sudoku.

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| `GetSudokus()` | 51/95/100 grilles | Trois niveaux de difficulte (Easy/Medium/Hard) |
| `TestSolvers()` | Performance multi-solveurs | Execution parallele avec timeout |
| `DisplayResults()` | Graphiques Plotly.NET | Visualisation des temps de resolution |
| `SolveSudoku()` | Test unitaire | Resolution individuelle avec affichage |

**Points cles** :
1. **Chargement intelligent** : Recherche recursive du dossier `Puzzles` dans l'arborescence
2. **Robustesse** : Gestion des timeouts (3000ms par defaut) et exceptions
3. **Mesures** : Temps d'execution total + nombre de grilles resolues
4. **Disqualification** : Un solver echouant sur une grille est disqualifie pour la difficulte

> **Note technique** : La methode `TestSolvers` utilise `Interlocked.Increment` pour un thread-safe increment du compteur de solutions. Le `CancellationToken` permet d'interrompre proprement les solveurs trop lents.

## Exercice : Validation d'une grille Sudoku

### Enonce

Implementez une methode `IsValidSolution` qui verifie qu'une grille est une solution valide de Sudoku, c'est-a-dire que chaque ligne, chaque colonne et chaque bloc 3x3 contient exactement une fois chaque chiffre de 1 a 9.

Utilisez cette methode pour valider les resultats de `SudokuHelper.SolveSudoku`.

### Indices

- Parcourez les 9 lignes, 9 colonnes et 9 blocs
- Pour chaque unite, verifiez que les 9 chiffres sont tous presents sans doublon
- `SudokuGrid.AllNeighbours` contient deja les indices des unites

Exercice a completer


## Resume et perspectives

Ce notebook a pose les fondations de toute la serie Sudoku en definissant trois composants essentiels. La classe `SudokuGrid` encapsule la representation d'une grille 9x9 avec le pre-calcul des voisins (`AllNeighbours`, `CellNeighbours`), ce qui evite les recalculs couteux lors de la resolution. L'interface `ISudokuSolver` implante le pattern Strategie, permettant de permuter les algorithmes de resolution sans modifier le code client. Enfin, la classe `SudokuHelper` fournit une infrastructure de benchmark complete avec chargement de puzzles, mesures de performance et visualisation Plotly.NET.

L'infrastructure de test (`TestSolvers`, `DisplayResults`) permet de comparer objectivement les solveurs sur trois niveaux de difficulte (Easy, Medium, Hard) avec gestion des timeouts et des disqualifications. Ce cadre de benchmark sera utilise dans tous les notebooks suivants pour mesurer les performances de chaque algorithme.

Le notebook suivant, [Sudoku-1-Backtracking](Sudoku-1-Backtracking-Csharp.ipynb), utilise ces classes pour implementer le premier algorithme de resolution : le backtracking recursif avec ses heuristiques d'amelioration.

## Affichage des Puzzles de chaque Difficulté

Nous allons charger et afficher un puzzle de chaque niveau de difficulté : Facile, Moyen et Difficile.


In [2]:
// Chargement et affichage d'un puzzle facile
var easySudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Easy).FirstOrDefault();
display($"Puzzle Facile:\n{easySudoku}");

// Chargement et affichage d'un puzzle moyen
var mediumSudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Medium).FirstOrDefault();
display($"Puzzle Moyen:\n{mediumSudoku}");

// Chargement et affichage d'un puzzle difficile
var hardSudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Hard).FirstOrDefault();
display($"Puzzle Difficile:\n{hardSudoku}");

Puzzle Facile:
-------------------------------
| 9     2 |       5 | 4     3 | 
| 1       |    6  3 |    2  5 | 
| 5     8 | 4     7 |    6    | 
-------------------------------
|    2  6 | 3     9 |       1 | 
|    5  7 |    1    | 2  9    | 
|    9    | 6  7    | 5  3    | 
-------------------------------
| 2  4    | 5  3    | 6       | 
| 7     5 | 2       | 3     4 | 
|    8    |    4  1 | 9  5    | 
-------------------------------

Puzzle Moyen:
-------------------------------
| 8  5    |       2 | 4       | 
| 7  2    |         |       9 | 
|       4 |         |         | 
-------------------------------
|         | 1     7 |       2 | 
| 3     5 |         | 9       | 
|    4    |         |         | 
-------------------------------
|         |    8    |    7    | 
|    1  7 |         |         | 
|         |    3  6 |    4    | 
-------------------------------

Puzzle Difficile:
-------------------------------
| 4       |         | 8     5 | 
|    3    |         |         | 
|         | 7       |         | 
-------------------------------
|    2    |         |    6    | 
|         |    8    | 4       | 
|         |    1    |         | 
-------------------------------
|         | 6     3 |    7    | 
| 5       | 2       |         | 
| 1     4 |         |         | 
-------------------------------

### Interprétation : Structure des Puzzles Sudoku

**Sortie obtenue** : Trois puzzles de difficulté croissante ont été chargés et affichés. La différence de complexité se voit visuellement par le nombre de cases pré-remplies.

| Difficulté | Cases vides (estimation) | Observation visuelle |
|------------|------------------------|---------------------|
| Facile | ~30-35 | Plus de 50% des cases sont déjà remplies, beaucoup de contraintes visibles |
| Moyen | ~45-50 | Environ 50% de cases vides, structure moins évidente |
| Difficile | ~55-60 | Très peu de cases pré-remplies (environ 40%), contraintes minimales |

**Points clés** :
1. **Échantillonnage représentatif** : La méthode `GetSudokus().FirstOrDefault()` retourne le premier puzzle disponible de chaque niveau, ce qui permet une comparaison consistante.
2. **Corrélation difficulté/densité** : Plus le puzzle a de cases vides, plus l'espace de recherche est grand et plus le backtracking devra explorer de combinaisons.
3. **Regles du jeu respectees** : Chaque puzzle respecte les contraintes du Sudoku (uniques chiffres 1-9 par ligne, colonne, bloc 3x3).
4. **Preparation aux tests** : Ces trois puzzles serviront de base pour évaluer les performances du solveur de backtracking.

> **Note technique** : Les puzzles sont générés algorithmiquement pour garantir qu'ils ont exactement une solution unique. Cette propriété est cruciale pour évaluer la complétude de l'algorithme de backtracking.

**Impact sur la performance** : Le nombre de cases vides determine directement la taille de l'arbre de recherche du backtracking. Avec 60 cases vides, le pire cas théorique est 9^60 possibilités, mais les contraintes reduisent drastiquement ce nombre en pratique.

## Code du solver en C#

Nous allons maintenant implémenter ce solveur en C#.

### Classe `BacktrackingDotNetSolver`

In [3]:
public class BacktrackingDotNetSolver : ISudokuSolver
{
    public SudokuGrid Solve(SudokuGrid s)
    {
        callCount = 0;
        Search(s, 0, 0);
        Console.WriteLine($"BacktrackingDotNetSolver: {callCount} search calls");
        return s;
    }

    private int callCount = 0;

    private bool Search(SudokuGrid s, int row, int col)
    {
        callCount++;
        if (row == 9) return true;
        if (col == 9) return Search(s, row + 1, 0);
        if (s.Cells[row, col] != 0) return Search(s, row, col + 1);

        for (int num = 1; num <= 9; num++)
        {
            if (IsValid(s, row, col, num))
            {
                s.Cells[row, col] = num;
                if (Search(s, row, col + 1)) return true;
                s.Cells[row, col] = 0;
            }
        }
        return false;
    }

    private bool IsValid(SudokuGrid s, int row, int col, int val)
    {
        for (int i = 0; i < 9; i++)
            if (s.Cells[row, i] == val || s.Cells[i, col] == val)
                return false;

        int startRow = 3 * (row / 3), startCol = 3 * (col / 3);
        for (int i = 0; i < 3; i++)
            for (int j = 0; j < 3; j++)
                if (s.Cells[startRow + i, startCol + j] == val)
                    return false;

        return true;
    }
}
Console.WriteLine("BacktrackingDotNetSolver class defined");


BacktrackingDotNetSolver class defined


## Test du Solveur

Nous allons maintenant tester notre solveur de Sudoku par backtracking en utilisant une grille de Sudoku.


## Exercice : Vérifier qu'une grille est valide

**Objectif :**
Implémentez une méthode qui vérifie si une grille complètement remplie
respecte toutes les contraintes du Sudoku (lignes, colonnes, blocs uniques).

**Indice :**
Parcourez chaque ligne, colonne et bloc 3x3 et vérifiez que chaque ensemble
contient exactement les chiffres 1 à 9 sans doublon.


In [1]:
// EXERCICE : Verifier qu'une grille est valide
public bool IsGridValid(int[,] grid)
{
    // TODO: Verifiez que chaque ligne, colonne et bloc 3x3
    // contient les chiffres 1-9 sans doublon
    return false; // TODO etudiant
}


Exercice a completer


In [4]:
BacktrackingDotNetSolver solver = new BacktrackingDotNetSolver();

// Test du puzzle facile
var easySudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Easy).FirstOrDefault();
Console.WriteLine("Puzzle Sudoku Facile Initial:");
SudokuHelper.SolveSudoku(easySudoku, solver);

// Test du puzzle moyen
var mediumSudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Medium).FirstOrDefault();
Console.WriteLine("Puzzle Sudoku Moyen Initial:");
SudokuHelper.SolveSudoku(mediumSudoku, solver);

// Test du puzzle difficile
var hardSudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Hard).FirstOrDefault();
Console.WriteLine("Puzzle Sudoku Difficile Initial:");
SudokuHelper.SolveSudoku(hardSudoku, solver);



Puzzle Sudoku Facile Initial:


Résolution par le solver BacktrackingDotNetSolver du Sudoku:
 -------------------------------
| 9     2 |       5 | 4     3 | 
| 1       |    6  3 |    2  5 | 
| 5     8 | 4     7 |    6    | 
-------------------------------
|    2  6 | 3     9 |       1 | 
|    5  7 |    1    | 2  9    | 
|    9    | 6  7    | 5  3    | 
-------------------------------
| 2  4    | 5  3    | 6       | 
| 7     5 | 2       | 3     4 | 
|    8    |    4  1 | 9  5    | 
-------------------------------

BacktrackingDotNetSolver: 122 search calls


Sudoku renvoyé:
-------------------------------
| 9  6  2 | 1  8  5 | 4  7  3 | 
| 1  7  4 | 9  6  3 | 8  2  5 | 
| 5  3  8 | 4  2  7 | 1  6  9 | 
-------------------------------
| 8  2  6 | 3  5  9 | 7  4  1 | 
| 3  5  7 | 8  1  4 | 2  9  6 | 
| 4  9  1 | 6  7  2 | 5  3  8 | 
-------------------------------
| 2  4  9 | 5  3  8 | 6  1  7 | 
| 7  1  5 | 2  9  6 | 3  8  4 | 
| 6  8  3 | 7  4  1 | 9  5  2 | 
-------------------------------
Nombre d'erreurs réstantes: 0
Temps de résolution: 0,4354 ms

Puzzle Sudoku Moyen Initial:


...

Sudoku renvoyé:
-------------------------------
| 8  5  9 | 6  1  2 | 4  3  7 | 
| 7  2  3 | 8  5  4 | 1  6  9 | 
| 1  6  4 | 3  7  9 | 5  2  8 | 
-------------------------------
| 9  8  6 | 1  4  7 | 3  5  2 | 
| 3  7  5 | 2  6  8 | 9  1  4 | 
| 2  4  1 | 5  9  3 | 7  8  6 | 
-------------------------------
| 4  3  2 | 9  8  1 | 6  7  5 | 
| 6  1  7 | 4  2  5 | 8  9  3 | 
| 5  9  8 | 7  3  6 | 2  4  1 | 
-------------------------------
Nombre d'erreurs réstantes: 0
Temps de résolution: 141,0824 ms

Puzzle Sudoku Difficile Initial:


Résolution par le solver BacktrackingDotNetSolver du Sudoku:
 -------------------------------
| 4       |         | 8     5 | 
|    3    |         |         | 
|         | 7       |         | 
-------------------------------
|    2    |         |    6    | 
|         |    8    | 4       | 
|         |    1    |         | 
-------------------------------
|         | 6     3 |    7    | 
| 5       | 2       |         | 
| 1     4 |         |         | 
-------------------------------

BacktrackingDotNetSolver: 12625368 search calls


Sudoku renvoyé:
-------------------------------
| 4  1  7 | 3  6  9 | 8  2  5 | 
| 6  3  2 | 1  5  8 | 9  4  7 | 
| 9  5  8 | 7  2  4 | 3  1  6 | 
-------------------------------
| 8  2  5 | 4  3  7 | 1  6  9 | 
| 7  9  1 | 5  8  6 | 4  3  2 | 
| 3  4  6 | 9  1  2 | 7  5  8 | 
-------------------------------
| 2  8  9 | 6  4  3 | 5  7  1 | 
| 5  7  3 | 2  9  1 | 6  8  4 | 
| 1  6  4 | 8  7  5 | 2  9  3 | 
-------------------------------
Nombre d'erreurs réstantes: 0
Temps de résolution: 4127,7309 ms

### Interprétation : Performance du Backtracking

**Sortie obtenue** : Trois puzzles de difficulté croissante ont été résolus avec succès. Le nombre d'appels récursifs et le temps de résolution augmentent exponentiellement avec la difficulté.

| Difficulté | Appels récursifs | Temps (ms) | Facteur d'augmentation |
|------------|-----------------|------------|------------------------|
| Facile | 122 | 0,4354 | 1x (reference) |
| Moyen | 490 304 | 141,0824 | 4 019x |
| Difficile | 12 625 368 | 4 127,7309 | 103 489x |

**Points clés** :
1. **Complexité exponentielle** : Le nombre d'appels récursifs explose littéralement entre le puzzle facile (122 appels) et le puzzle difficile (12,6 millions d'appels).
2. **Efficacité sur les puzzles simples** : Le backtracking naïve est très rapide (moins d'une demi-millisecond) pour les puzzles faciles.
3. **Limites sur les puzzles difficiles** : Plus de 4 secondes pour un puzzle difficile difficilement acceptable pour une application interactive.
4. **Tous les puzzles résolus** : L'algorithme trouve toujours une solution (complétude), mais le coût en temps varie énormément.

> **Note technique** : Le temps de résolution n'est pas linéaire par rapport au nombre d'appels récursifs. Le puzzle difficile nécessite 25 fois plus d'appels que le puzzle moyen, mais prend 29 fois plus de temps. Cela s'explique par le coût de la gestion de la pile d'appels et des opérations de validation. (Les temps sont non-déterministes : ils dépendent de la machine et de la charge CPU au moment de l'exécution.)

**Comparaison avec la théorie** : L'analyse des résultats confirme les propriétés théoriques du backtracking :
- **Complétude** : Tous les puzzles sont résolus avec 0 erreurs restantes
- **Cout** : Exponentiel dans le pire cas, mais acceptable pour les instances simples
- **Amélioration nécessaire** : Les heuristiques (MRV, Forward Checking) sont indispensables pour les puzzles difficiles

## Exercice : Comparer backtracking simple et MRV

**Objectif :**
Comparez les performances du solveur simple et du solveur MRV sur
des puzzles de différentes difficultés.

**Indice :**
Utilisez un Stopwatch pour mesurer le temps et comptez les appels récursifs.


In [1]:
// EXERCICE : Comparer backtracking simple et MRV
public Dictionary<string, (double TimeMs, int Calls)> CompareSolvers(int[,] puzzle)
{
    // TODO: Lancez les deux solveurs sur le meme puzzle et comparez
    // les temps de resolution et le nombre d'appels recursifs
    return null; // TODO etudiant
}


Exercice a completer


## Exercice (guidé) : Backtracking avec Heuristique MRV (Minimum Remaining Values)

### Énoncé

L'implémentation actuelle de `BacktrackingDotNetSolver` choisit les cellules a remplir dans l'ordre de parcours (de gauche à droite, de haut en bas). Implémentez une version améliorée avec l'heuristique **MRV** (Minimum Remaining Values) :

L'heuristique MRV choisit en priorité la cellule qui a le **moins de valeurs possibles** (le domaine le plus petit). Intuitivement, on commence par les cellules les plus contraintes pour détecter les échecs plus tot et réduire l'espace de recherche.

Implémentez `BacktrackingMRVSolver` :
1. Pour chaque cellule vide, calculez son domaine (valeurs 1-9 compatibles avec les contraintes)
2. Choisissez la cellule avec le plus petit domaine non vide
3. Si un domaine est vide, retournez `false` immédiatement (échec précoce)
4. Comparez le nombre d'appels récursifs avec `BacktrackingDotNetSolver`

**Indice :**

Le calcul du domaine consiste a retirer de {1..9} toutes les valeurs déjà présentes dans la meme ligne, colonne ou bloc. L'heuristique MRV réduit drastiquement les appels récursifs sur les puzzles difficiles.

In [5]:
// Exemple guide : BacktrackingMRVSolver avec heuristique MRV
// TODO: Implementez un solveur de backtracking utilisant l'heuristique MRV

public class BacktrackingMRVSolver : ISudokuSolver
{
    private int callCount = 0;
    
    public SudokuGrid Solve(SudokuGrid s)
    {
        callCount = 0;
        var grid = (SudokuGrid)s.Clone();
        Search(grid);
        Console.WriteLine($"BacktrackingMRVSolver: {callCount} search calls");
        return grid;
    }
    
    private HashSet<int> GetDomain(SudokuGrid s, int row, int col)
    {
        // TODO: Retourne l'ensemble des valeurs valides pour la cellule (row, col)
        // Retirez de {1..9} toutes les valeurs deja presentes dans :
        // - la ligne row
        // - la colonne col
        // - le bloc 3x3 contenant (row, col)
        return new HashSet<int>();  // TODO etudiant : a completer
    }
    
    private (int row, int col) SelectMRVCell(SudokuGrid s)
    {
        // TODO: Trouver la cellule vide avec le plus petit domaine
        // Parcourez toutes les cellules vides, calculez leur domaine
        // et retournez la cellule dont le domaine est minimal
        // Retournez (-1, -1) si aucune cellule vide n'existe (grille complete)
        return (-1, -1);  // TODO etudiant : a completer
    }
    
    private bool Search(SudokuGrid s)
    {
        callCount++;
        
        // TODO: Algorithme de backtracking avec MRV
        // Etape 1: Choisir la cellule avec le plus petit domaine via SelectMRVCell
        // Etape 2: Si aucune cellule vide, la grille est complete -> return true
        // Etape 3: Si le domaine est vide, echec -> return false
        // Etape 4: Pour chaque valeur du domaine, essayer et recurser
        // Etape 5: Si la recursion echoue, restaurer et essayer la valeur suivante
        return false;  // TODO etudiant : a completer
    }
}

// Test de votre implementation (decommentez quand BacktrackingMRVSolver est implemente)
// var mrvSolver = new BacktrackingMRVSolver();
// var hardSudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Hard).FirstOrDefault();
// var results = SudokuHelper.TestSolvers(new List<(string, ISudokuSolver)>
// {
//     ("Backtracking simple", new BacktrackingDotNetSolver()),
//     ("Backtracking MRV", mrvSolver)
// });
// SudokuHelper.DisplayResults(results);

Console.WriteLine("TODO: Implementez BacktrackingMRVSolver pour comparer les performances");

TODO: Implementez BacktrackingMRVSolver pour comparer les performances


## Exercice : Compter le nombre de solutions

**Objectif :**
Modifiez le solveur backtracking pour compter le nombre total de solutions
d'un puzzle donne (en arrêtant après un maximum pour eviter les boucles infinies).

**Indice :**
Ajoutez un compteur global et arrêtez la recherche après `maxCount` solutions trouvées.


In [1]:
// EXERCICE : Compter le nombre de solutions
public int CountSolutions(int[,] puzzle, int maxCount = 100)
{
    // TODO: Modifiez le backtracking pour compter les solutions
    // au lieu de s'arreter a la premiere trouvee
    return 0; // TODO etudiant
}


Exercice a completer


## Conclusion et Analyse des Performances

L'algorithme de backtracking est une méthode efficace pour résoudre des puzzles de Sudoku simples a modérés. Pour des puzzles plus complexes, il peut devenir lent en raison du grand nombre de combinaisons possibles.

| Aspect | Observation |
|--------|-------------|
| **Complétude** | Oui - trouve toujours une solution si elle existe |
| **Optimalité** | N/A (il n'y a qu'une solution par grille valide) |
| **Complexité** | O(9^81) dans le pire cas, beaucoup mieux en pratique |
| **Forces** | Simple a implémenter, garanti de trouver une solution |
| **Faiblesses** | Lent sur les puzzles difficiles sans heuristiques |

### Pistes d'amélioration

- **MRV (Minimum Remaining Values)** : choisir la cellule la plus contrainte en premier
- **Forward Checking** : éliminer les valeurs impossibles après chaque assignation
- **Propagation de contraintes** : voir [Sudoku-7 Norvig](Sudoku-7-Norvig-Csharp.ipynb) pour une approche plus sophistiquée

### Prochaines étapes

Dans les notebooks suivants, nous explorerons des techniques plus avancées :
- [Sudoku-3 Genetic](Sudoku-3-Genetic-Csharp.ipynb) : approche métaheuristique
- [Sudoku-10 OR-Tools](Sudoku-10-ORTools-Csharp.ipynb) : programmation par contraintes industrielle

---

**Navigation** : [<< Sudoku-0 Environment](Sudoku-0-Environment-Csharp.ipynb) | [Index](README.md) | [Sudoku-3 Genetic >>](Sudoku-3-Genetic-Csharp.ipynb)
